In [ ]:
!pip install timm opencv-python scikit-learn matplotlib

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d orvile/complete-blood-count-cbc-dataset
!unzip complete-blood-count-cbc-dataset.zip -d cbc_dataset

In [ ]:
import os, cv2, glob, random, shutil
import xml.etree.ElementTree as ET
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from tqdm.notebook import tqdm

import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [ ]:
class_map = {
    "WBC": 0,
    "RBC": 1,
    "Platelets": 2
}

In [ ]:
def convert_xml_to_yolo(xml_file, output_file):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        size = root.find('size')
        if size is None:
            return

        w = int(size.find('width').text)
        h = int(size.find('height').text)

        lines = []

        for obj in root.iter('object'):
            cls = obj.find('name').text

            if cls not in class_map:
                continue

            xmlbox = obj.find('bndbox')
            if xmlbox is None:
                continue

            xmin = int(xmlbox.find('xmin').text)
            xmax = int(xmlbox.find('xmax').text)
            ymin = int(xmlbox.find('ymin').text)
            ymax = int(xmlbox.find('ymax').text)

            # clamp values (important)
            xmin = max(0, xmin)
            ymin = max(0, ymin)
            xmax = min(w, xmax)
            ymax = min(h, ymax)

            bw = xmax - xmin
            bh = ymax - ymin

            if bw <= 0 or bh <= 0:
                continue

            x_center = ((xmin + xmax) / 2) / w
            y_center = ((ymin + ymax) / 2) / h
            width = bw / w
            height = bh / h

            cls_id = class_map[cls]
            lines.append(f"{cls_id} {x_center} {y_center} {width} {height}")

        if len(lines) > 0:
            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            with open(output_file, "w") as f:
                f.write("\n".join(lines))

    except Exception as e:
        print(f"Error in {xml_file}: {e}")

In [ ]:
xml_files = sorted(glob.glob(
    "cbc_dataset/Complete-Blood-Cell-Count-Dataset-master/*/Annotations/*.xml",
    recursive=True
))

for xml in xml_files:
    txt_path = xml.replace("Annotations", "labels").replace(".xml", ".txt")
    convert_xml_to_yolo(xml, txt_path)

In [ ]:
import os

train_imgs = sorted(glob.glob("cbc_dataset/**/Training/**/*.jpg", recursive=True))
val_imgs   = sorted(glob.glob("cbc_dataset/**/Validation/**/*.jpg", recursive=True))

# ambil filename saja
train_names = set(os.path.basename(p) for p in train_imgs)
val_names   = set(os.path.basename(p) for p in val_imgs)

# cek overlap
overlap = train_names.intersection(val_names)

print(f"Total val images: {len(val_names)}")
print(f"Overlap with train: {len(overlap)}")
print("Sample overlap:", list(overlap)[:10])

Validation Set sama Percis dengan Training jadi Leakage

In [ ]:
train_imgs = sorted(glob.glob("cbc_dataset/**/Training/**/*.jpg", recursive=True))
test_imgs  = sorted(glob.glob("cbc_dataset/**/Testing/**/*.jpg", recursive=True))

all_imgs = train_imgs + test_imgs

print("Train original:", len(train_imgs))
print("Test original:", len(test_imgs))
print("Total used:", len(all_imgs))

In [ ]:
image_paths = all_imgs

valid_data = []

for img in image_paths:
    label = img.replace("Images", "labels").replace(".jpg", ".txt")

    if os.path.exists(label) and os.path.getsize(label) > 0:
        valid_data.append((img, label))

print(f"Total valid samples: {len(valid_data)}")

In [ ]:
import os
from collections import Counter

# ambil hanya nama file image
filenames = [os.path.basename(img) for img, _ in valid_data]

# hitung frekuensi
counter = Counter(filenames)

# cari yang duplicate
duplicates = [name for name, count in counter.items() if count > 1]

print(f"Total duplicate filenames: {len(duplicates)}")
print("Sample duplicates:", duplicates[:10])

In [ ]:
train_data, temp_data = train_test_split(
    valid_data,
    test_size=0.3,
    random_state=42,
    shuffle=True
)

val_data, test_data = train_test_split(
    temp_data,
    test_size=1/3,
    random_state=42,
    shuffle=True
)

print(f"Train: {len(train_data)}")
print(f"Val: {len(val_data)}")
print(f"Test: {len(test_data)}")

In [ ]:
base_dir = "yolo_dataset"

for split in ["train", "val", "test"]:
    os.makedirs(f"{base_dir}/images/{split}", exist_ok=True)
    os.makedirs(f"{base_dir}/labels/{split}", exist_ok=True)

In [ ]:
def copy_data(data, split):
    for img, label in data:
        shutil.copy(img, f"{base_dir}/images/{split}/{os.path.basename(img)}")
        shutil.copy(label, f"{base_dir}/labels/{split}/{os.path.basename(label)}")

copy_data(train_data, "train")
copy_data(val_data, "val")
copy_data(test_data, "test")

In [ ]:
def count_classes(data):
    counter = Counter()

    for _, label in data:
        with open(label, "r") as f:
            for line in f:
                cls = int(line.split()[0])
                counter[cls] += 1

    return counter

print("Train:", count_classes(train_data))
print("Val:", count_classes(val_data))
print("Test:", count_classes(test_data))

train_counter = count_classes(train_data)
val_counter   = count_classes(val_data)
test_counter  = count_classes(test_data)

all_classes = sorted(set(train_counter) | set(val_counter) | set(test_counter))

train_values = [train_counter.get(c, 0) for c in all_classes]
val_values   = [val_counter.get(c, 0) for c in all_classes]
test_values  = [test_counter.get(c, 0) for c in all_classes]

x = range(len(all_classes))
width = 0.25

plt.figure()

plt.bar([i - width for i in x], train_values, width=width, label='Train')
plt.bar(x, val_values, width=width, label='Validation')
plt.bar([i + width for i in x], test_values, width=width, label='Test')

plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Class Distribution (Dynamic)')
plt.xticks(x, all_classes)
plt.legend()

plt.show()

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Define class mapping
CLASS_NAMES = {
    0: "WBC",
    1: "RBC",
    2: "Platelets"
}

# Fixed class order
ALL_CLASSES = [0, 1, 2]

CLASS_COLORS = {
    0: "#1f77b4",
    1: "#ff7f0e",
    2: "#2ca02c",
}

def count_classes(data):
    counter = Counter()

    for _, label in data:
        with open(label, "r") as f:
            for line in f:
                cls = int(line.split()[0])
                counter[cls] += 1

    return counter

def make_autopct(counter):
    total = sum(counter.values())

    def autopct(pct):
        count = int(round(pct * total / 100.0))
        return f"{pct:.1f}%\n({count})"

    return autopct

def plot_pie(ax, counter, title):
    # Ensure consistent class order
    labels = [CLASS_NAMES[c] for c in ALL_CLASSES]
    sizes = [counter.get(c, 0) for c in ALL_CLASSES]
    colors = [CLASS_COLORS[c] for c in ALL_CLASSES]

    wedges, texts, autotexts = ax.pie(
        sizes,
        labels=labels,
        colors=colors,
        autopct=make_autopct(counter),
        startangle=45,
        textprops={'fontsize': 10}
    )

    ax.set_title(title, fontsize=12)
    ax.axis('equal')

    return wedges

# Count objects
train_counter = count_classes(train_data)
val_counter   = count_classes(val_data)
test_counter  = count_classes(test_data)


fig = plt.figure(figsize=(10, 8))
gs = gridspec.GridSpec(2, 2)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, :])

wedges = plot_pie(ax1, train_counter, "Train Set")
plot_pie(ax2, val_counter, "Validation Set")
plot_pie(ax3, test_counter, "Test Set")

fig.legend(
    wedges,
    [CLASS_NAMES[c] for c in ALL_CLASSES],
    loc="lower center",
    ncol=3,
    fontsize=11
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

In [ ]:
import os

def get_filenames(data):
    return set(os.path.basename(img) for img, _ in data)

train_names = get_filenames(train_data)
val_names   = get_filenames(val_data)
test_names  = get_filenames(test_data)

train_val_overlap = train_names & val_names
train_test_overlap = train_names & test_names
val_test_overlap = val_names & test_names

print("=== FILENAME CHECK ===")
print(f"Train-Val overlap: {len(train_val_overlap)}")
print(f"Train-Test overlap: {len(train_test_overlap)}")
print(f"Val-Test overlap: {len(val_test_overlap)}")

if train_val_overlap:
    print("Sample overlap:", list(train_val_overlap))

In [ ]:
import hashlib
from PIL import Image
from tqdm import tqdm

# function untuk hash gambar
def get_image_hash(img_path):
    try:
        with Image.open(img_path) as img:
            img = img.convert("RGB")
            img = img.resize((256, 256))  # normalize
            return hashlib.md5(img.tobytes()).hexdigest()
    except:
        return None


train_imgs = [img for img, _ in train_data]
val_imgs   = [img for img, _ in val_data]
test_imgs  = [img for img, _ in test_data]


train_hashes = set(filter(None, [get_image_hash(p) for p in tqdm(train_imgs)]))

val_hashes = set(filter(None, [get_image_hash(p) for p in tqdm(val_imgs)]))

test_hashes = set(filter(None, [get_image_hash(p) for p in tqdm(test_imgs)]))


# check overlap
train_val_overlap = train_hashes & val_hashes
train_test_overlap = train_hashes & test_hashes
val_test_overlap = val_hashes & test_hashes

print("\n=== OVERLAP CHECK ===")
print(f"Train-Val overlap: {len(train_val_overlap)}")
print(f"Train-Test overlap: {len(train_test_overlap)}")
print(f"Val-Test overlap: {len(val_test_overlap)}")

In [ ]:
import os

counts = []

def check_split(split):
    img_dir = f"yolo_dataset/images/{split}"
    label_dir = f"yolo_dataset/labels/{split}"

    img_count = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    label_count = len(os.listdir(label_dir)) if os.path.exists(label_dir) else 0

    print(f"{split.upper()}:")
    print(f"Images: {img_count}")
    print(f"Labels: {label_count}")
    print(f"Match: {img_count == label_count}\n")
    counts.append({"split": split, "images": img_count})

for s in ["train", "val", "test"]:
    check_split(s)

print(counts)

In [ ]:
import matplotlib.pyplot as plt

sizes = [c["images"] for c in counts]
labels = [f"{c['split']} ({c['images']})" for c in counts]

if sum(sizes) > 0:
    plt.figure()
    plt.pie(sizes, labels=labels, autopct='%1.1f%%')
    plt.title("Dataset Split Distribution (Images)")
    plt.show()
else:
    print("No data to plot.")

In [ ]:
yaml_content = """
path: /content

train: yolo_dataset/images/train
val: yolo_dataset/images/val
test: yolo_dataset/images/test

nc: 3
names: ['WBC', 'RBC', 'Platelets']
"""

with open("wbc.yaml", "w") as f:
    f.write(yaml_content)

wandb initialization

In [ ]:
!pip install -U ultralytics wandb
!yolo settings wandb=True

In [ ]:
import wandb

In [ ]:
!rm -rf /content/runs/detect/CBC-Blood-Cell-Detection

Helper

In [ ]:
def print_excel_row(model_name, val, test):
    print(f"{model_name}\t"
          f"{val.box.mr:.4f}\t"
          f"{val.box.mp:.4f}\t"
          f"{val.box.map50:.4f}\t"
          f"{val.box.map:.4f}\t"
          f"{test.box.mr:.4f}\t"
          f"{test.box.mp:.4f}\t"
          f"{test.box.map50:.4f}\t"
          f"{test.box.map:.4f}")

**LR TUNING**

YOLO11s Baseline (LR: 0.001)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

model.train(
    data="wbc.yaml",
    epochs=60,
    imgsz=640,
    batch=8,

    lr0=0.001,
    # augment OFF (minimal)
    mosaic=0.0,
    mixup=0.0,
    scale=0.0,
    fliplr=0.0,

    optimizer="AdamW",
    patience=15,
    project="AoL-DL-1",
    name="11s_baseline",
    exist_ok=True,
)

In [ ]:
val_metrics = model.val(data="wbc.yaml", split="val")
test_metrics = model.val(data="wbc.yaml", split="test")

print("Model\tVal Recall\tVal Precision\tVal mAP50\tVal mAP50-95\tTest Recall\tTest Precision\tTest mAP50\tTest mAP50-95")
print_excel_row("YOLO11s_LR_0.001", val_metrics, test_metrics)

YOLO11s  LR:0.0005

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

model.train(
    data="wbc.yaml",
    epochs=60,
    imgsz=640,
    batch=8,

    lr0=0.0005,
    # augment OFF (minimal)
    mosaic=0.0,
    mixup=0.0,
    scale=0.0,
    fliplr=0.0,

    optimizer="AdamW",
    patience=15,
    project="AoL-DL-1",
    name="11s_lr_0.0005",
    exist_ok=True,
)

In [ ]:
val_metrics = model.val(data="wbc.yaml", split="val")
test_metrics = model.val(data="wbc.yaml", split="test")

print("Model\tVal Recall\tVal Precision\tVal mAP50\tVal mAP50-95\tTest Recall\tTest Precision\tTest mAP50\tTest mAP50-95")
print_excel_row("YOLO11s_LR_0.0005", val_metrics, test_metrics)

YOLO11s LR:0.0001

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

model.train(
    data="wbc.yaml",
    epochs=60,
    imgsz=640,
    batch=8,

    lr0=0.0001,
    # augment OFF (minimal)
    mosaic=0.0,
    mixup=0.0,
    scale=0.0,
    fliplr=0.0,

    optimizer="AdamW",
    patience=15,
    project="AoL-DL-1",
    name="11s_lr_0.0001",
    exist_ok=True,
)

In [ ]:
val_metrics = model.val(data="wbc.yaml", split="val")
test_metrics = model.val(data="wbc.yaml", split="test")

print("Model\tVal Recall\tVal Precision\tVal mAP50\tVal mAP50-95\tTest Recall\tTest Precision\tTest mAP50\tTest mAP50-95")
print_excel_row("YOLO11s_LR_0.0001", val_metrics, test_metrics)

YOLO11s LR:0.00001

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

model.train(
    data="wbc.yaml",
    epochs=60,
    imgsz=640,
    batch=8,

    lr0=0.00001,
    # augment OFF (minimal)
    mosaic=0.0,
    mixup=0.0,
    scale=0.0,
    fliplr=0.0,

    optimizer="AdamW",
    patience=15,
    project="AoL-DL-1",
    name="11s_lr_0.00001",
    exist_ok=True,
)

In [ ]:
val_metrics = model.val(data="wbc.yaml", split="val")
test_metrics = model.val(data="wbc.yaml", split="test")

print("Model\tVal Recall\tVal Precision\tVal mAP50\tVal mAP50-95\tTest Recall\tTest Precision\tTest mAP50\tTest mAP50-95")
print_excel_row("YOLO11s_LR_0.00001", val_metrics, test_metrics)

In [ ]:
import shutil

shutil.make_archive(
    "Experiments (LR-Augment)",
    'zip',
    "runs/detect/AoL-DL-1"
)

In [ ]:
from ultralytics import YOLO

models = {
    "LR_0.001": "/content/runs/detect/AoL-DL-1/11s_baseline/weights/best.pt",
    "LR_0.0005": "/content/runs/detect/AoL-DL-1/11s_lr_0.0005/weights/best.pt",
    "LR_0.0001": "/content/runs/detect/AoL-DL-1/11s_lr_0.0001/weights/best.pt",
    "LR_0.00001": "/content/runs/detect/AoL-DL-1/11s_lr_0.00001/weights/best.pt"
}

results = []

for name, path in models.items():
    model = YOLO(path)

    metrics = model.val(data="wbc.yaml", split="val", verbose=False)

    results.append({
        "Model": name,
        "Precision": metrics.box.mp,
        "Recall": metrics.box.mr,
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map
    })

print("Model\tPrecision\tRecall\tmAP50\tmAP50-95")
for r in results:
    print(f"{r['Model']}\t{r['Precision']:.4f}\t{r['Recall']:.4f}\t{r['mAP50']:.4f}\t{r['mAP50-95']:.4f}")

Best Overall LR = 0.00001

**Augmentation**

YOLO11s Baseline with Augment

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

model.train(
    data="wbc.yaml",
    epochs=60,
    imgsz=640,
    batch=8,

    lr0=0.001,
    # Recommended dari YOLO
    mosaic=1.0,
    mixup=0.0,
    scale=0.5,
    fliplr=0.5,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    optimizer="AdamW",
    patience=15,
    project="AoL-DL-1",
    name="11s_baseline_augment",
    exist_ok=True,
)

In [ ]:
val_metrics = model.val(data="wbc.yaml", split="val")
test_metrics = model.val(data="wbc.yaml", split="test")

print("Model\tVal Recall\tVal Precision\tVal mAP50\tVal mAP50-95\tTest Recall\tTest Precision\tTest mAP50\tTest mAP50-95")
print_excel_row("YOLO11s_baseline_augment", val_metrics, test_metrics)

YOLO11s Baseline Augment Strong

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

model.train(
    data="wbc.yaml",
    epochs=60,
    imgsz=640,
    batch=8,

    lr0=0.001,
    # Recommended dari YOLO
    mosaic=1.0,
    mixup=0.1,
    scale=0.7,
    fliplr=0.5,

    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,

    optimizer="AdamW",
    patience=15,
    project="AoL-DL-1",
    name="11s_baseline_aug_strong",
    exist_ok=True,
)

In [ ]:
val_metrics = model.val(data="wbc.yaml", split="val")
test_metrics = model.val(data="wbc.yaml", split="test")

print("Model\tVal Recall\tVal Precision\tVal mAP50\tVal mAP50-95\tTest Recall\tTest Precision\tTest mAP50\tTest mAP50-95")
print_excel_row("YOLO11s_baseline_augment_strong", val_metrics, test_metrics)

In [ ]:
from ultralytics import YOLO

models = {
    "No Augment": "/content/runs/detect/AoL-DL-1/11s_baseline/weights/best.pt",
    "Recommended Augment": "/content/runs/detect/AoL-DL-1/11s_baseline_augment/weights/best.pt",
    "Strong Augment": "/content/runs/detect/AoL-DL-1/11s_baseline_aug_strong/weights/best.pt",
}

results = []

for name, path in models.items():
    model = YOLO(path)

    metrics = model.val(data="wbc.yaml", split="val", verbose=False)

    results.append({
        "Model": name,
        "Precision": metrics.box.mp,
        "Recall": metrics.box.mr,
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map
    })

print("Model\tPrecision\tRecall\tmAP50\tmAP50-95")
for r in results:
    print(f"{r['Model']}\t{r['Precision']:.4f}\t{r['Recall']:.4f}\t{r['mAP50']:.4f}\t{r['mAP50-95']:.4f}")

Hyperparameter Tuning

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

model.tune(
    data="wbc.yaml",
    epochs=10,
    iterations=10,
    optimizer="AdamW",
    patience=15,
    project="AoL-DL-1-YOLO11-Tuning",
    name="11s_tuning",
    exist_ok=True,
    plots=True,
    save=True
)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

model.train(
    data="wbc.yaml",
    epochs=60,
    imgsz=640,
    batch=8,

    lr0= 0.00933,
    lrf= 0.02425,
    momentum= 0.89447,
    weight_decay= 0.00048,
    warmup_epochs= 3.16919,
    warmup_momentum= 0.95,
    box= 13.44393,
    cls= 0.35442,
    cls_pw= 0.01013,
    dfl= 1.36377,
    hsv_h= 0.01657,
    hsv_s= 0.88519,
    hsv_v= 0.37149,
    degrees= 0.00073,
    translate= 0.15369,
    scale= 0.32029,
    shear= 0.02421,
    perspective= 0.0,
    flipud= 0.0053,
    fliplr= 0.56732,
    bgr= 0.00336,
    mosaic= 1.0,
    mixup= 0.00324,
    cutmix= 0.00126,
    copy_paste= 0.00291,
    close_mosaic= 8,

    optimizer="AdamW",
    patience=15,
    project="AoL-DL-1",
    name="11s_baseline_aug_strong",
    exist_ok=True,
)

In [ ]:
val_metrics = model.val(data="wbc.yaml", split="val")
test_metrics = model.val(data="wbc.yaml", split="test")

print("Model\tVal Recall\tVal Precision\tVal mAP50\tVal mAP50-95\tTest Recall\tTest Precision\tTest mAP50\tTest mAP50-95")
print_excel_row("YOLO11s_baseline_augment_strong", val_metrics, test_metrics)

Try Best LR and Augment (Recommended)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

model.train(
    data="wbc.yaml",
    epochs=100,
    imgsz=640,
    batch=8,

    lr0=0.00001,
    # Recommended dari YOLO
    mosaic=1.0,
    mixup=0.0,
    scale=0.5,
    fliplr=0.5,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    optimizer="AdamW",
    patience=15,
    project="AoL-DL-1",
    name="11s_lr_augment",
    exist_ok=True,
)

In [ ]:
val_metrics = model.val(data="wbc.yaml", split="val")
test_metrics = model.val(data="wbc.yaml", split="test")

print("Model\tVal Recall\tVal Precision\tVal mAP50\tVal mAP50-95\tTest Recall\tTest Precision\tTest mAP50\tTest mAP50-95")
print_excel_row("YOLO11s_lr_augment", val_metrics, test_metrics)

Test Best LR with Strong Augmentation

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")

model.train(
    data="wbc.yaml",
    epochs=100,
    imgsz=640,
    batch=8,

    lr0=0.00001,

    mosaic=1.0,
    mixup=0.1,
    scale=0.7,
    fliplr=0.5,

    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.5,

    optimizer="AdamW",
    patience=15,
    project="AoL-DL-1",
    name="11s_lr_strong_augment",
    exist_ok=True,
)

In [ ]:
val_metrics = model.val(data="wbc.yaml", split="val")
test_metrics = model.val(data="wbc.yaml", split="test")

print("Model\tVal Recall\tVal Precision\tVal mAP50\tVal mAP50-95\tTest Recall\tTest Precision\tTest mAP50\tTest mAP50-95")
print_excel_row("YOLO11s_lr_str_augment", val_metrics, test_metrics)

In [ ]:
import shutil

shutil.make_archive(
    "LR_Augment",
    'zip',
    "runs/detect/AoL-DL-1"
)

from google.colab import files

files.download("LR_Augment.zip")

**YOLOv8 Comparison**

Load YOLO11s Best

In [ ]:
from google.colab import files
files.upload()

In [ ]:
from ultralytics import YOLO

model_11s = YOLO("/content/best.pt")

In [ ]:
val_metrics_11s = model_11s.val(data="wbc.yaml", split="val")
test_metrics_11s = model_11s.val(data="wbc.yaml", split="test")

print("Model\tVal Recall\tVal Precision\tVal mAP50\tVal mAP50-95\tTest Recall\tTest Precision\tTest mAP50\tTest mAP50-95")
print_excel_row("YOLO11s_BEST", val_metrics_11s, test_metrics_11s)

Cek Sama Seblumnya

*   YOLO11s_lr_augment	0.9130	0.8856	0.9330	0.6319	0.8988	0.8684	0.9386	0.6701
*   YOLO11s_BEST	0.9130	0.8856	0.9330	0.6319	0.8988	0.8684	0.9386	0.6701

Sama Load Aman (Reproducibility aman)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")

model.train(
    data="wbc.yaml",
    epochs=100,
    imgsz=640,
    batch=8,

    lr0=0.00001,
    # Recommended dari YOLO
    mosaic=1.0,
    mixup=0.0,
    scale=0.5,
    fliplr=0.5,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    optimizer="AdamW",
    patience=15,
    project="AoL-DL-1",
    name="v8s_tuned_comparison",
    exist_ok=True,
)

In [ ]:
val_metrics = model.val(data="wbc.yaml", split="val")
test_metrics = model.val(data="wbc.yaml", split="test")

print("Model\tVal Recall\tVal Precision\tVal mAP50\tVal mAP50-95\tTest Recall\tTest Precision\tTest mAP50\tTest mAP50-95")
print_excel_row("YOLOv8s", val_metrics, test_metrics)

Comparison
*   YOLO11s 0.9130	0.8856	0.9330	0.6319	0.8988	0.8684	0.9386	0.6701
*   YOLOv8s 0.9129	0.8564	0.9174	0.6190	0.9188	0.8008	0.9092	0.6366



In [ ]:
import shutil

shutil.make_archive(
    "v8s_comparison",
    'zip',
    "runs/detect/AoL-DL-1"
)

from google.colab import files

files.download("v8s_comparison.zip")

YOLO11s Tuned vs YOLOv8s Tuned (On Test Set)

In [ ]:
def extract_metrics(metrics):
    return {
        "Precision": metrics.box.mp,
        "Recall": metrics.box.mr,
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map
    }

v8 = extract_metrics(test_metrics)
v11 = extract_metrics(test_metrics_11s)

models = ["YOLOv8s", "YOLO11s"]
metric_names = list(v8.keys())

v8_values = list(v8.values())
v11_values = list(v11.values())

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2, v8_values, width, label="YOLOv8s")
bars2 = ax.bar(x + width/2, v11_values, width, label="YOLO11s")

ax.set_xticks(x)
ax.set_xticklabels(metric_names)

ax.set_title("Test Set Comparison (YOLOv8s vs YOLO11s)")
ax.set_ylabel("Score")
ax.legend()

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2,
                height,
                f"{height:.3f}",
                ha='center',
                va='bottom')

plt.tight_layout()
plt.show()

YOLO11s better performance than YOLOv8s with same parameters

In [ ]:
import matplotlib.pyplot as plt

results = model.predict(
    source="yolo_dataset/images/test",
    conf=0.5,
    save=True
)

img_path = results[0].path
img = plt.imread(img_path)

plt.imshow(img)
plt.axis("off")

In [ ]:
best_model = YOLO("/content/best.pt")

In [ ]:
class_names = ['WBC', 'RBC', 'Platelets']

def visualize_gt_vs_pred(image_path, model):
    img = cv2.imread(image_path)
    h, w, _ = img.shape

    label_path = image_path.replace("images", "labels").replace(".jpg", ".txt")

    # 🔹 Draw Ground Truth (GREEN)
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                cls, x, y, bw, bh = map(float, line.split())

                x1 = int((x - bw/2) * w)
                y1 = int((y - bh/2) * h)
                x2 = int((x + bw/2) * w)
                y2 = int((y + bh/2) * h)

                label = class_names[int(cls)]

                cv2.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 2)
                cv2.putText(
                    img,
                    f"GT: {label}",
                    (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0,255,0),
                    2
                )

    # 🔹 Draw Predictions (RED)
    results = model(img)[0]

    boxes = results.boxes.xyxy
    scores = results.boxes.conf
    classes = results.boxes.cls

    for box, score, cls in zip(boxes, scores, classes):
        x1, y1, x2, y2 = map(int, box)
        label = class_names[int(cls)]

        cv2.rectangle(img, (x1, y1), (x2, y2), (0,0,255), 2)
        cv2.putText(
            img,
            f"{label} {score:.2f}",
            (x1, y2 + 15),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0,0,255),
            2
        )

    plt.figure(figsize=(8,8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")

In [ ]:
visualize_gt_vs_pred(
    "/content/yolo_dataset/images/test/BloodImage_00003.jpg",
    model
)

In [ ]:
import shutil

shutil.make_archive(
    "predictions",
    'zip',
    "runs/detect/predict"
)

from google.colab import files

files.download("predictions.zip")

Load Best Yolo Model

In [ ]:
from google.colab import files
files.upload()

In [ ]:
yolo_model = YOLO("/content/best.pt")

Try Improve YOLO

In [ ]:
import ultralytics
print(ultralytics.__file__)

In [ ]:
conv_path = "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/conv.py"

CBAM

In [ ]:
conv_path = "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/conv.py"

cbam_code = """
# ================= CBAM =================
import torch
import torch.nn as nn

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        return x * self.sigmoid(
            self.fc(self.avg_pool(x)) + self.fc(self.max_pool(x))
        )

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))

class CBAM(nn.Module):
    def __init__(self, c1):
        super().__init__()
        self.ca = ChannelAttention(c1)
        self.sa = SpatialAttention()

    def forward(self, x):
        return self.sa(self.ca(x))
# =======================================
"""

with open(conv_path, "a") as f:
    f.write(cbam_code)

print("✅ CBAM added to conv.py")

In [ ]:
init_path = "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/__init__.py"

with open(init_path, "a") as f:
    f.write("\nfrom .conv import CBAM\n")

print("✅ CBAM registered")

In [ ]:
tasks_path = "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py"

with open(tasks_path, "r") as f:
    content = f.read()

# Add CBAM import at top
if "CBAM" not in content:
    content = content.replace(
        "from ultralytics.nn.modules",
        "from ultralytics.nn.modules import CBAM\nfrom ultralytics.nn.modules"
    )

with open(tasks_path, "w") as f:
    f.write(content)

print("✅ CBAM imported into tasks.py")

In [ ]:
from ultralytics.nn.modules.conv import CBAM
print("CBAM loaded:", CBAM)

In [ ]:
!cp /usr/local/lib/python3.*/dist-packages/ultralytics/cfg/models//11/yolo11.yaml ./yolo11_cbam.yaml

In [ ]:
yaml_content = """
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Ultralytics YOLO11 object detection model with P3/8 - P5/32 outputs
# Model docs: https://docs.ultralytics.com/models/yolo11
# Task docs: https://docs.ultralytics.com/tasks/detect

# Parameters
# YOLO11s + CBAM

nc: 3

scales: # model compound scaling constants, i.e. 'model=yolo11n.yaml' will call yolo11.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.50, 0.25, 1024] # summary: 181 layers, 2624080 parameters, 2624064 gradients, 6.6 GFLOPs
  s: [0.50, 0.50, 1024] # summary: 181 layers, 9458752 parameters, 9458736 gradients, 21.7 GFLOPs
  m: [0.50, 1.00, 512] # summary: 231 layers, 20114688 parameters, 20114672 gradients, 68.5 GFLOPs
  l: [1.00, 1.00, 512] # summary: 357 layers, 25372160 parameters, 25372144 gradients, 87.6 GFLOPs
  x: [1.00, 1.50, 512] # summary: 357 layers, 56966176 parameters, 56966160 gradients, 196.0 GFLOPs

backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]]           # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]]          # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]   # 2
  - [-1, 1, Conv, [256, 3, 2]]          # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]   # 4
  - [-1, 1, Conv, [512, 3, 2]]          # 5-P4/16
  - [-1, 2, C3k2, [512, True]]          # 6
  - [-1, 1, Conv, [1024, 3, 2]]         # 7-P5/32
  - [-1, 2, C3k2, [1024, True]]         # 8
  - [-1, 1, SPPF, [1024, 5]]            # 9
  - [-1, 2, C2PSA, [1024]]              # 10
  - [-1, 1, CBAM, [512]]                # 11

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] # cat backbone P4
  - [-1, 2, C3k2, [512, False]] # 13

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] # cat backbone P3
  - [-1, 2, C3k2, [256, False]] # 16 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 14], 1, Concat, [1]] # cat head P4
  - [-1, 2, C3k2, [512, False]] # 19 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] # cat head P5
  - [-1, 2, C3k2, [1024, True]] # 22 (P5/32-large)

  - [[17, 20, 22], 1, Detect, [nc]] # Detect(P3, P4, P5)
"""

with open("yolo11s_cbam.yaml", "w") as f:
    f.write(yaml_content)

In [ ]:
yaml_content = """
path: /content

train: yolo_dataset/images/train
val: yolo_dataset/images/val
test: yolo_dataset/images/test

nc: 3
names: ['WBC', 'RBC', 'Platelets']
"""

with open("wbc.yaml", "w") as f:
    f.write(yaml_content)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s_cbam.yaml")

model.info()

In [ ]:
model = YOLO("yolo11s_cbam.yaml")
model.load("yolo11s.pt")

model.train(
    data="wbc.yaml",
    epochs=100,
    imgsz=640,
    batch=8,

    lr0=0.00001,
    # Recommended dari YOLO
    mosaic=1.0,
    mixup=0.0,
    scale=0.5,
    fliplr=0.5,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    optimizer="AdamW",
    patience=15,
    project="AoL-DL-1",
    name="yolo11s_cbam_akhir",
    exist_ok=True,
)

In [ ]:
model = YOLO("/content/runs/detect/AoL-DL-1/yolo11s_cbam_akhir/weights/best.pt")

In [ ]:
val_metrics = model.val(data="wbc.yaml", split="val")
test_metrics = model.val(data="wbc.yaml", split="test")

print("Model\tVal Recall\tVal Precision\tVal mAP50\tVal mAP50-95\tTest Recall\tTest Precision\tTest mAP50\tTest mAP50-95")
print_excel_row("YOLO11s_CBAM_AP4", val_metrics, test_metrics)

In [ ]:
import shutil

shutil.make_archive(
    "experiments",
    'zip',
    "runs/detect/CBC-Blood-Cell-Detection"
)

In [ ]:
import pandas as pd

print(results_cbam)
# df = pd.DataFrame(results_cbam)

# df.index = [
#     "Val mAP50",
#     "Val mAP50-95",
#     "Val Precision",
#     "Val Recall",
#     "Test mAP50",
#     "Test mAP50-95",
#     "Test Precision",
#     "Test Recall"
# ]

# df = df.round(4)
# df.to_csv("model_comparison_full.csv")
# print(df)

In [ ]:
model.info()

SE Block

In [ ]:
conv_path = "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/conv.py"

se_code = """
# ================= SE BLOCK =================
import torch
import torch.nn as nn

class SEBlock(nn.Module):
    def __init__(self, c1, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(c1, c1 // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(c1 // reduction, c1, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.fc(self.pool(x))
# ===========================================
"""

with open(conv_path, "a") as f:
    f.write(se_code)

print("✅ SEBlock added")

In [ ]:
from ultralytics.nn.modules.conv import SEBlock
from ultralytics.nn.tasks import DetectionModel
import ultralytics.nn.tasks as tasks

# Register in both places
DetectionModel.SEBlock = SEBlock
tasks.SEBlock = SEBlock

In [ ]:
print(SEBlock)

In [ ]:
yaml_content = """
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Ultralytics YOLO11 object detection model with P3/8 - P5/32 outputs
# Model docs: https://docs.ultralytics.com/models/yolo11
# Task docs: https://docs.ultralytics.com/tasks/detect

# Parameters
nc: 3 # number of classes
scales: # model compound scaling constants, i.e. 'model=yolo11n.yaml' will call yolo11.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.50, 0.25, 1024] # summary: 181 layers, 2624080 parameters, 2624064 gradients, 6.6 GFLOPs
  s: [0.50, 0.50, 1024] # summary: 181 layers, 9458752 parameters, 9458736 gradients, 21.7 GFLOPs
  m: [0.50, 1.00, 512] # summary: 231 layers, 20114688 parameters, 20114672 gradients, 68.5 GFLOPs
  l: [1.00, 1.00, 512] # summary: 357 layers, 25372160 parameters, 25372144 gradients, 87.6 GFLOPs
  x: [1.00, 1.50, 512] # summary: 357 layers, 56966176 parameters, 56966160 gradients, 196.0 GFLOPs

# YOLO11n backbone
backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]] # 5-P4/16
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, SEBlock, [256]]
  - [-1, 1, Conv, [1024, 3, 2]] # 7-P5/32
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]] # 9
  - [-1, 2, C2PSA, [1024]] # 10

# YOLO11n head
head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 2, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 2, C3k2, [512, False]]

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 2, C3k2, [1024, True]]

  - [[16, 19, 22], 1, Detect, [nc]]
"""

with open("yolo11s_se.yaml", "w") as f:
    f.write(yaml_content)

In [ ]:
yaml_content = """
path: /content

train: yolo_dataset/images/train
val: yolo_dataset/images/val
test: yolo_dataset/images/test

nc: 3
names: ['WBC', 'RBC', 'Platelets']
"""

with open("wbc.yaml", "w") as f:
    f.write(yaml_content)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11s_se.yaml")

model.load("yolo11s.pt")

model.train(
    data="wbc.yaml",
    epochs=60,
    imgsz=640,
    batch=8,

    lr0=0.0005,
    cls=1.0,

    optimizer="AdamW",
    patience=15,
    project="CBC-Blood-Cell-Detection",
    name="yolo11s_se_1",
    exist_ok=True,
)

In [ ]:
model = YOLO("/content/runs/detect/CBC-Blood-Cell-Detection/yolo11s_se_1/weights/best.pt")

In [ ]:
val_metrics = model.val(data="wbc.yaml", split="val")
test_metrics = model.val(data="wbc.yaml", split="test")

print("Model\tVal Recall\tVal Precision\tVal mAP50\tVal mAP50-95\tTest Recall\tTest Precision\tTest mAP50\tTest mAP50-95")
print_excel_row("YOLO11s_SE_1", val_metrics, test_metrics)

In [ ]:
import shutil

shutil.make_archive(
    "experiments_SE",
    'zip',
    "runs/detect/CBC-Blood-Cell-Detection"
)

In [ ]:
print(results_se)